<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/Music_Intellectual_Property1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -U langchain langchain-text-splitters sentence-transformers rank-bm25 faiss-cpu openai tiktoken google-generativeai

In [ ]:
import numpy as np
from typing import List, Dict
# Removed OpenAI import
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss


# Initialize Gemini Client
import os
from google.colab import userdata
import google.generativeai as genai # Corrected import statement

# Get Gemini API key from secrets and initialize
GoogleAIAPI = userdata.get('GoogleAIAPI')
genai.configure(api_key=GoogleAIAPI) # Should work now with correct import
gemini_model = genai.GenerativeModel('gemini-2.5-flash') # Using gemini-pro as a default, you can change this if needed

In [ ]:
# 1. MOCK KNOWLEDGE BASE (Reflecting accurate 2025/2026 UK IP Music Data)
documents = [
    {
        "id": "doc_1",
        "title": "Copyright, Designs and Patents Act 1988 (CDPA) - Sec 16",
        "text": "Under the CDPA 1988, copyright in a musical work is infringed by a person who without the licence of the copyright owner does, or authorises another to do, any of the acts restricted by the copyright. Restricted acts include copying the work, issuing copies to the public, and communicating the work to the public."
    },
    {
        "id": "doc_2",
        "title": "UK Government Report on Copyright and AI (March 18, 2026)",
        "text": "The UK government officially confirmed it has abandoned 'Option 3', a broad text and data mining (TDM) exception that would have permitted AI companies to use songs and recordings to train models without permission unless rightsholders opted out. The legal position remains unchanged: copyright material cannot be used for AI development and training without permission."
    },
    {
        "id": "doc_3",
        "title": "Warner Music v Crumbl (May 2026 Settlement)",
        "text": "Warner Music Group and cookie chain Crumbl reached a settlement in a $24M copyright infringement lawsuit over unauthorized use of music in TikTok posts. Crumbl was accused of misappropriating at least 159 sound recordings for promotional videos. WMG argued this deprived creators of compensation and demonstrated willful infringement, as Crumbl continued using trending audios after receiving a cease-and-desist."
    },
    {
        "id": "doc_4",
        "title": "Emotional Perception AI Ltd v Comptroller-General of Patents [2023] EWHC 2948",
        "text": "Justice Mann overturned the UKIPO's refusal to grant a patent for an AI system using artificial neural networks (ANNs) to recommend similar songs. The UKIPO argued it was a computer program 'as such' under Section 1(2)(c) of the Patents Act 1977. Justice Mann ruled an ANN operates autonomously based on acquired knowledge, not human-crafted instructions, and the notification of a music recommendation file constituted a 'technical effect'."
    },
    {
        "id": "doc_5",
        "title": "Getty Images v Stability AI (2025/2026 Appeal)",
        "text": "At trial in late 2025, Getty dropped primary copyright infringement claims because model training occurred in the US. Getty instead argued secondary infringement, claiming Stability AI imported an 'infringing copy' into the UK. The first instance judgment found against Getty, but an appeal on secondary copyright infringement was granted for 2026, which the music industry is monitoring closely regarding generative AI liability."
    }
]

# 2. CHUNKING (Semantic / Recursive)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = []
for doc in documents:
    splits = text_splitter.split_text(doc["text"])
    for split in splits:
        chunks.append({
            "id": doc["id"],
            "title": doc["title"],
            "text": split
        })

print(f"Created {len(chunks)} chunks from {len(documents)} documents.")

In [ ]:
# Load Embedding Model (Dense) and Cross-Encoder (Reranker)
print("Loading models (this may take a moment)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2') # Fast, local dense embedder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2') # Reranking model

# 1. Create Dense Vector Store (FAISS)
embeddings = embedder.encode([chunk["text"] for chunk in chunks])
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

# 2. Create Sparse Index (BM25)
tokenized_corpus = [chunk["text"].lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print("Indexing complete.")

In [ ]:
def reciprocal_rank_fusion(dense_ranks, sparse_ranks, k=60):
    """Combines BM25 and Vector search results using RRF."""
    rrf_scores = {}
    for rank, chunk_idx in enumerate(dense_ranks):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1 / (k + rank)
    for rank, chunk_idx in enumerate(sparse_ranks):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1 / (k + rank)

    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return sorted_indices

def retrieve_baseline(query, top_k=3):
    """Vanilla Vector Search."""
    query_vector = embedder.encode([query])
    _, indices = index.search(np.array(query_vector).astype('float32'), top_k)
    return [chunks[i] for i in indices[0]]

def retrieve_enhanced(query, top_k=3):
    """Hybrid Search + Cross-Encoder Reranking."""
    # 1. Dense Retrieval
    query_vector = embedder.encode([query])
    _, dense_indices = index.search(np.array(query_vector).astype('float32'), top_k*2)
    dense_indices = dense_indices[0].tolist()

    # 2. Sparse Retrieval
    tokenized_query = query.lower().split()
    sparse_scores = bm25.get_scores(tokenized_query)
    sparse_indices = np.argsort(sparse_scores)[::-1][:top_k*2].tolist()

    # 3. Fuse Results (RRF)
    fused_indices = reciprocal_rank_fusion(dense_indices, sparse_indices)[:top_k*2]
    candidate_chunks = [chunks[i] for i in fused_indices]

    # 4. Cross-Encoder Reranking
    cross_inp = [[query, chunk["text"]] for chunk in candidate_chunks]
    cross_scores = reranker.predict(cross_inp)

    # Sort by reranker score
    scored_chunks = list(zip(candidate_chunks, cross_scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, score in scored_chunks[:top_k]]

def generate_answer(query, retrieved_chunks):
    """Generates the grounded response using an LLM."""
    context = "\n\n".join([f"Source: {c['title']}\n{c['text']}" for c in retrieved_chunks])

    system_prompt = (
        "You are an expert UK Music IP Lawyer. Answer the user's query based ONLY on the provided legal context. "
        "Cite the specific case names, statutes, or dates provided. If the answer is not in the context, say 'I cannot advise based on the current legal context.'"
    )

    # Using Gemini API
    response = gemini_model.generate_content(
        [{"role": "user", "parts": [system_prompt, f"Context:\n{context}\n\nQuery: {query}"]}]
    )
    return response.text

In [ ]:
# Define diverse test queries and their target document IDs
test_set = [
    {
        "type": "Simple Fact",
        "query": "Did the UK government introduce a new text and data mining exception for AI in 2026?",
        "target_id": "doc_2"
    },
    {
        "type": "Deep Context",
        "query": "How did Justice Mann rule on artificial neural networks regarding the Patents Act 1977?",
        "target_id": "doc_4"
    },
    {
        "type": "Ambiguous Edge Case",
        "query": "Can a brand just use trending music on TikTok to promote their cookies?",
        "target_id": "doc_3"
    },
    {
        "type": "Keyword heavy",
        "query": "What are the rules on secondary infringement and importing copies as seen in the Stability AI case?",
        "target_id": "doc_5"
    }
]

print("--- EVALUATION RUN ---")
for test in test_set:
    q = test["query"]
    target = test["target_id"]

    # Retrieve
    base_chunks = retrieve_baseline(q)
    enh_chunks = retrieve_enhanced(q)

    # Calc Retrieval Success (Recall@3)
    base_success = any(c["id"] == target for c in base_chunks)
    enh_success = any(c["id"] == target for c in enh_chunks)

    # Generate using Enhanced
    answer = generate_answer(q, enh_chunks)

    print(f"\n[{test['type']}] Query: {q}")
    print(f"Retrieval Match -> Baseline: {'✅' if base_success else '❌'} | Enhanced: {'✅' if enh_success else '❌'}")
    print(f"Enhanced Answer: {answer}")